# Llama 3.2 Benchmarks v5 — Fixed Grader + MATH Token Budget

**Purpose:** Fix two bugs discovered in v4's pipeline that were suppressing scores across all 5 models:

1. **GSM8K grader false negatives:** Strict string match rejected `pred='57.00'` vs `gold='57'`. Fix: numeric-equality check.
2. **MATH truncation:** `max_new_tokens=768` cut off ~25% of Llama-3B's reasoning chains mid-proof, before the `\boxed{}` final answer. Fix: bump to 1536.

## Two-part structure

### Part A — Grader re-score (seconds, no GPU needed)

Reads the existing `*__gsm8k.records.json` files from Drive, re-grades them with the improved numeric grader, overwrites the `*__gsm8k.json` summary files. No model runs. This gives you an immediate read on how much the GSM8K grader bug was suppressing scores.

### Part B — MATH re-run (~40 min on Blackwell)

Re-runs all 5 models on MATH with `max_new_tokens=1536`. This is the only way to fix the truncation bug — the records can't be re-graded because the generations themselves were cut off. Overwrites the `*__math.json` files.

## Files touched

- **Re-scored (Part A):** `{Llama-1B, Llama-3B, Phi-3, Gemma-2-2B, TinyLlama}__gsm8k.json` — summary files only, records preserved
- **Overwritten (Part B):** Same 5 models × `__math.json` + `__math.records.json`
- **Untouched:** All `__bfcl.json` files, all `__bfcl.records.json` files

## Expected impact

| Model | GSM8K current | GSM8K after Part A (est) | MATH current | MATH after Part B (est) |
|---|---|---|---|---|
| TinyLlama | 2.81% | ~3% | 0.20% | ~1% |
| Llama-3.2-1B | 39.12% | ~40-41% | 24.40% | ~28-32% |
| Gemma-2-2B | 58.38% | ~60-61% | 14.80% | ~18-22% |
| Llama-3.2-3B | 72.48% | ~74-75% | 40.20% | ~50-54% |
| Phi-3-mini | 87.19% | ~88-89% | 41.20% | ~55-60% |

## Runtime setup

Requires Blackwell / H100 for Part B. Part A works on any CPU. If you want to run just Part A to inspect the grader fix, skip the GPU check and start from `## 3. Part A`.


---
## 1. Setup


In [1]:
!pip install -q \
    'transformers>=4.45.0,<5.0' \
    'accelerate>=0.34.0' \
    'datasets>=2.20.0' \
    'sentencepiece' 'protobuf' \
    'huggingface_hub>=0.24' \
    'sympy>=1.12' 'antlr4-python3-runtime==4.11' \
    'tqdm' 'pandas'
print('[OK] deps installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 80.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
[OK] deps installed


In [2]:
import torch

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f'Device: {name}')
    print(f'VRAM:   {vram} GB')
else:
    print('[INFO] No GPU — Part A (re-scoring) will work, Part B (MATH re-run) will not.')


CUDA: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM:   101.97 GB


In [3]:
# HF auth — needed for Part B (model downloads). Skip if only running Part A.
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (hf_...): ').strip()
login(token=HF_TOKEN, add_to_git_credential=False)
print('[OK] logged in')


HF token (hf_...): ··········
[OK] logged in


In [4]:
# Mount Drive — same results dir as v4 and fill-in
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
RESULTS_DIR = Path('/content/drive/MyDrive/llama32_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)

existing = sorted(p.name for p in RESULTS_DIR.glob('*.json')
                  if not p.name.endswith('.records.json'))
print(f'\n{len(existing)} existing result files:')
for f in existing:
    print(f'  {f}')


Mounted at /content/drive
Results dir: /content/drive/MyDrive/llama32_results

20 existing result files:
  Llama-3.2-1B-Instruct__bfcl.json
  Llama-3.2-1B-Instruct__gsm8k.json
  Llama-3.2-1B-Instruct__math.json
  Llama-3.2-1B-Instruct__nexus.json
  Llama-3.2-3B-Instruct__bfcl.json
  Llama-3.2-3B-Instruct__gsm8k.json
  Llama-3.2-3B-Instruct__math.json
  Llama-3.2-3B-Instruct__nexus.json
  Phi-3-mini-4k-instruct__bfcl.json
  Phi-3-mini-4k-instruct__gsm8k.json
  Phi-3-mini-4k-instruct__math.json
  Phi-3-mini-4k-instruct__nexus.json
  TinyLlama-1.1B-Chat-v1.0__bfcl.json
  TinyLlama-1.1B-Chat-v1.0__gsm8k.json
  TinyLlama-1.1B-Chat-v1.0__math.json
  TinyLlama-1.1B-Chat-v1.0__nexus.json
  gemma-2-2b-it__bfcl.json
  gemma-2-2b-it__gsm8k.json
  gemma-2-2b-it__math.json
  gemma-2-2b-it__nexus.json


---
## 2. Config


In [5]:
import os, json, re, time, gc, math

# The 5 models we benchmark
ALL_MODELS = [
    'meta-llama/Llama-3.2-1B-Instruct',
    'meta-llama/Llama-3.2-3B-Instruct',
    'microsoft/Phi-3-mini-4k-instruct',
    'google/gemma-2-2b-it',
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
]

# For Part B — which models to re-run MATH on. Default: all 5.
# Set to a subset like ['meta-llama/Llama-3.2-3B-Instruct'] to test on one first.
PART_B_MODELS = ALL_MODELS

# MATH config — THE KEY FIX
N_MATH                = 500
MAX_NEW_TOKENS_MATH   = 1536     # was 768 — doubled to avoid truncation
BATCH_SIZE            = 16       # Drop to 8 on A100 40GB

# Models whose chat template raises on role='system' — merge into user turn
NO_SYSTEM_ROLE = {'google/gemma-2-2b-it'}

# Per-model SDPA settings
MODEL_LOAD_KWARGS = {
    'meta-llama/Llama-3.2-1B-Instruct': {'attn_implementation': 'sdpa'},
    'meta-llama/Llama-3.2-3B-Instruct': {'attn_implementation': 'sdpa'},
    'microsoft/Phi-3-mini-4k-instruct': {'attn_implementation': 'sdpa'},
    'google/gemma-2-2b-it': {'attn_implementation': 'sdpa'},
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0': {'attn_implementation': 'sdpa'},
}

PARAMS_B = {
    'meta-llama/Llama-3.2-1B-Instruct': 1.23,
    'meta-llama/Llama-3.2-3B-Instruct': 3.21,
    'microsoft/Phi-3-mini-4k-instruct': 3.82,
    'google/gemma-2-2b-it': 2.61,
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0': 1.10,
}

print('Part B will re-run MATH on:')
for m in PART_B_MODELS:
    print(f'  {m}')
print(f'\nMAX_NEW_TOKENS_MATH: {MAX_NEW_TOKENS_MATH}  (was 768 in v4)')


Part B will re-run MATH on:
  meta-llama/Llama-3.2-1B-Instruct
  meta-llama/Llama-3.2-3B-Instruct
  microsoft/Phi-3-mini-4k-instruct
  google/gemma-2-2b-it
  TinyLlama/TinyLlama-1.1B-Chat-v1.0

MAX_NEW_TOKENS_MATH: 1536  (was 768 in v4)


---
## 3. Part A — GSM8K grader re-score (no GPU needed)

Reads each model's `__gsm8k.records.json`, re-grades with the improved grader, overwrites the `__gsm8k.json` summary. The records files are left unchanged so you can always re-grade again with a different rule if needed.

### The grader fix

**Old grader (v4):**
```python
ok = (pred is not None) and (pred == gold)   # string equality
```

**New grader:**
```python
def _grade_gsm8k(pred, gold):
    if pred is None or gold is None: return False
    # 1. Fast path: exact string match
    if pred == gold: return True
    # 2. Numeric equality (handles '57' vs '57.00', '$5' vs '5', etc)
    try:
        p = float(str(pred).replace(',', '').replace('$', ''))
        g = float(str(gold).replace(',', '').replace('$', ''))
        return abs(p - g) < 1e-6
    except Exception:
        return False
```

Expected impact: +2-3 pp per model. This is a pure grader change — **no model outputs change, no generations re-run**.


In [6]:
def _grade_gsm8k(pred, gold):
    '''Improved GSM8K grader: numeric equality with string fallback.'''
    if pred is None or gold is None:
        return False
    # Fast path: exact string match
    if pred == gold:
        return True
    # Numeric equality
    try:
        p = float(str(pred).replace(',', '').replace('$', '').strip())
        g = float(str(gold).replace(',', '').replace('$', '').strip())
        return abs(p - g) < 1e-6
    except Exception:
        return False


def regrade_gsm8k_file(summary_path, records_path):
    '''Re-score a single model's GSM8K records and update its summary file.
    Returns (model_slug, old_acc, new_acc, n).'''
    with open(records_path) as f:
        records = json.load(f)
    with open(summary_path) as f:
        summary = json.load(f)

    old_correct = sum(1 for r in records if r.get('correct'))
    old_acc = old_correct / max(len(records), 1)

    # Re-grade
    new_correct = 0
    for r in records:
        ok = _grade_gsm8k(r.get('pred'), r.get('gold'))
        r['correct'] = ok
        new_correct += int(ok)
    new_acc = new_correct / max(len(records), 1)

    # Update summary
    summary['accuracy'] = new_acc
    summary['n'] = len(records)
    summary['grader_version'] = 'v5_numeric_equality'

    # Write back
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    with open(records_path, 'w') as f:
        json.dump(records, f)

    return summary_path.stem, old_acc, new_acc, len(records)


print('=== Part A: GSM8K re-score ===\n')
print(f'{"Model":40s} {"n":>6s} {"Old":>8s} {"New":>8s} {"Delta":>8s}')
print('-' * 74)

results = []
for model_id in ALL_MODELS:
    slug = model_id.split('/')[-1]
    summary_path = RESULTS_DIR / f'{slug}__gsm8k.json'
    records_path = RESULTS_DIR / f'{slug}__gsm8k.records.json'

    if not records_path.exists():
        print(f'{slug:40s} {"--":>6s}    [SKIP: no records file]')
        continue
    if not summary_path.exists():
        print(f'{slug:40s} {"--":>6s}    [SKIP: no summary file]')
        continue

    _, old_acc, new_acc, n = regrade_gsm8k_file(summary_path, records_path)
    delta = new_acc - old_acc
    results.append((slug, old_acc, new_acc, n))
    print(f'{slug:40s} {n:>6d} {100*old_acc:>7.2f}% {100*new_acc:>7.2f}% {100*delta:>+7.2f}pp')

print('\n[DONE] GSM8K re-scoring complete — summary files updated.')
print('       Records files preserved (just the `correct` field updated).')


=== Part A: GSM8K re-score ===

Model                                         n      Old      New    Delta
--------------------------------------------------------------------------
Llama-3.2-1B-Instruct                      1319   39.12%   39.65%   +0.53pp
Llama-3.2-3B-Instruct                      1319   72.48%   74.60%   +2.12pp
Phi-3-mini-4k-instruct                     1319   87.19%   88.10%   +0.91pp
gemma-2-2b-it                              1319   58.38%   62.70%   +4.32pp
TinyLlama-1.1B-Chat-v1.0                   1319    2.81%    2.88%   +0.08pp

[DONE] GSM8K re-scoring complete — summary files updated.
       Records files preserved (just the `correct` field updated).


---
## 4. Part B — MATH re-run with larger token budget

Re-runs MATH for each model in `PART_B_MODELS` with `max_new_tokens=1536`. Overwrites `*__math.json` and `*__math.records.json`.

**If you only want to run Part A and stop here,** you can skip Part B. The existing MATH numbers will be ~5-15 pp below the published Meta numbers, but Part A alone gives you better GSM8K results with no additional compute.

**To run Part B on only one model first (smoke test),** change `PART_B_MODELS` in the config cell above. I recommend testing on `'meta-llama/Llama-3.2-3B-Instruct'` first (~5 min) to confirm the fix works before committing ~40 min to all 5 models.


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm


def load_model(model_id):
    print(f'\n[LOAD] {model_id}')
    t0 = time.time()
    extra = MODEL_LOAD_KWARGS.get(model_id, {}).copy()

    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        low_cpu_mem_usage=True,
        **extra,
    )
    model.eval()
    dt = time.time() - t0
    vram = torch.cuda.memory_allocated() / 1e9
    print(f'[LOAD] done in {dt:.1f}s | VRAM {vram:.2f} GB '
          f'| attn={extra.get("attn_implementation", "default")}')
    return tok, model


def free_model(tok, model):
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    vram = torch.cuda.memory_allocated() / 1e9
    print(f'[FREE] VRAM: {vram:.2f} GB')


In [8]:
def _build_messages(model_id, system, user):
    '''Gemma doesnt allow system role — merge into user turn.'''
    if model_id in NO_SYSTEM_ROLE:
        merged = f'{system.strip()}\n\n{user.strip()}' if system else user
        return [{'role': 'user', 'content': merged}]
    return [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user},
    ]


def _term_ids(tok):
    ids = [tok.eos_token_id]
    for special in ['<|eot_id|>', '<|end|>', '<end_of_turn>', '</s>']:
        try:
            tid = tok.convert_tokens_to_ids(special)
            if isinstance(tid, int) and tid != tok.unk_token_id:
                ids.append(tid)
        except Exception:
            pass
    return list({i for i in ids if i is not None})


def _apply_template(tok, model_id, system, user):
    msgs = _build_messages(model_id, system, user)
    try:
        return tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception as e:
        if 'system' in str(e).lower() and len(msgs) > 1:
            merged = f'{msgs[0]["content"]}\n\n{msgs[1]["content"]}'
            return tok.apply_chat_template(
                [{'role': 'user', 'content': merged}],
                tokenize=False, add_generation_prompt=True)
        raise


def chat_generate_batch(model, tok, model_id, systems, users, max_new_tokens):
    '''Batched greedy generation. Returns (texts, total_s, per_sample_s).'''
    assert len(systems) == len(users)
    bsz = len(systems)

    prompts = [_apply_template(tok, model_id, s, u)
               for s, u in zip(systems, users)]

    tok.padding_side = 'left'
    enc = tok(prompts, return_tensors='pt', padding=True, truncation=True,
              max_length=2048).to(model.device)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None,
            pad_token_id=tok.eos_token_id,
            eos_token_id=_term_ids(tok),
        )
    dt = time.time() - t0

    input_len = enc['input_ids'].shape[1]
    gen_ids = out[:, input_len:]
    texts = [t.strip() for t in tok.batch_decode(gen_ids, skip_special_tokens=True)]
    return texts, dt, dt / bsz


In [9]:
from datasets import load_dataset

MATH_SYSTEM = (
    'You are an expert mathematician. Solve the problem step by step using '
    'careful reasoning. Put your final answer inside \\boxed{...}.'
)


def _last_boxed(text):
    if not text: return None
    idx = text.rfind('\\boxed')
    if idx < 0: return None
    i = text.find('{', idx)
    if i < 0: return None
    depth, j = 1, i + 1
    while j < len(text) and depth > 0:
        if text[j] == '{':   depth += 1
        elif text[j] == '}': depth -= 1
        j += 1
    return text[i+1:j-1] if depth == 0 else None


def _norm_math(ans):
    if ans is None: return ''
    s = ans.strip()
    for t in ['\\!','\\,','\\;','\\:','\\ ',' ','\n','\r']:
        s = s.replace(t, '')
    s = s.replace('dfrac','frac').replace('tfrac','frac')
    s = re.sub(r'\\text\{[^}]*\}', '', s)
    s = s.replace('\\%','').replace('%','').replace('\\$','').replace('$','')
    return s.rstrip('.')


def _math_equal(pred, gold):
    if pred is None or gold is None: return False
    if _norm_math(pred) == _norm_math(gold): return True
    try:
        import sympy as sp
        from sympy.parsing.latex import parse_latex
        return bool(sp.simplify(parse_latex(pred) - parse_latex(gold)) == 0)
    except Exception:
        return False


def run_math(model, tok, model_id, n=N_MATH, batch_size=BATCH_SIZE,
             max_new_tokens=MAX_NEW_TOKENS_MATH):
    ds = load_dataset('HuggingFaceH4/MATH-500', split='test').select(range(min(n, 500)))
    examples = list(ds)
    correct, records, total_time = 0, [], 0.0

    for i in tqdm(range(0, len(examples), batch_size),
                  desc=f'MATH (bs={batch_size}, max_new={max_new_tokens})'):
        chunk = examples[i:i+batch_size]
        systems = [MATH_SYSTEM] * len(chunk)
        users = [ex['problem'] for ex in chunk]
        gens, dt_batch, dt_per = chat_generate_batch(
            model, tok, model_id, systems, users, max_new_tokens=max_new_tokens)
        total_time += dt_batch

        for ex, gen in zip(chunk, gens):
            pred = _last_boxed(gen)
            gold = ex.get('answer') or _last_boxed(ex.get('solution', ''))
            ok = _math_equal(pred, gold)
            correct += int(ok)
            records.append({'problem': ex['problem'], 'gold': gold,
                            'pred': pred, 'generation': gen,
                            'correct': ok, 'latency_s': dt_per})

    # Track truncation rate as diagnostic
    n_no_boxed = sum(1 for r in records if not r.get('pred'))

    return {'benchmark': 'MATH_0shot_CoT', 'n': len(records),
            'accuracy': correct / len(records),
            'mean_latency_s': total_time / len(records),
            'total_time_s': total_time,
            'batch_size': batch_size,
            'max_new_tokens': max_new_tokens,
            'n_no_boxed': n_no_boxed,
            'truncation_rate': n_no_boxed / len(records),
            'grader_version': 'v5',
            'records': records}


In [10]:
def _save(results, model_id, bench_name):
    slug = model_id.split('/')[-1]
    summary_path = RESULTS_DIR / f'{slug}__{bench_name}.json'
    records_path = RESULTS_DIR / f'{slug}__{bench_name}.records.json'

    summary = {k: v for k, v in results.items() if k != 'records'}
    summary['model_id'] = model_id

    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    with open(records_path, 'w') as f:
        json.dump(results.get('records', []), f)
    return summary_path.name


# Announce overwrites up-front
print('=== Part B: MATH re-run ===\n')
print(f'MAX_NEW_TOKENS_MATH = {MAX_NEW_TOKENS_MATH}  (up from 768 in v4)\n')
print('Files that will be overwritten:')
for m in PART_B_MODELS:
    slug = m.split('/')[-1]
    p = RESULTS_DIR / f'{slug}__math.json'
    status = 'OVERWRITE' if p.exists() else 'NEW'
    print(f'  [{status:9s}] {p.name}')
print()

overall_t0 = time.time()

for model_id in PART_B_MODELS:
    tok, model = load_model(model_id)
    torch.cuda.reset_peak_memory_stats()
    try:
        print(f'\n=== {model_id}  |  MATH (max_new={MAX_NEW_TOKENS_MATH}) ===')
        t_start = time.time()
        results = run_math(model, tok, model_id)
        t_elapsed = time.time() - t_start

        acc = results.get('accuracy')
        trunc_rate = results.get('truncation_rate', 0)
        print(f'  accuracy:        {100*acc:.2f}%  (n={results["n"]})')
        print(f'  truncation rate: {100*trunc_rate:.1f}%  (no \\boxed found)')
        print(f'  mean_latency:    {1000*results["mean_latency_s"]:.1f} ms/sample')
        print(f'  wall time:       {t_elapsed:.1f}s')
        peak_vram = torch.cuda.max_memory_allocated() / 1e9
        results['peak_vram_gb'] = peak_vram
        print(f'  peak_vram:       {peak_vram:.2f} GB')

        fname = _save(results, model_id, 'math')
        print(f'  saved -> {fname}')
    finally:
        free_model(tok, model)

overall_t = time.time() - overall_t0
print(f'\n[DONE] Part B total wall time: {overall_t/60:.1f} min')


=== Part B: MATH re-run ===

MAX_NEW_TOKENS_MATH = 1536  (up from 768 in v4)

Files that will be overwritten:
  [OVERWRITE] Llama-3.2-1B-Instruct__math.json
  [OVERWRITE] Llama-3.2-3B-Instruct__math.json
  [OVERWRITE] Phi-3-mini-4k-instruct__math.json
  [OVERWRITE] gemma-2-2b-it__math.json
  [OVERWRITE] TinyLlama-1.1B-Chat-v1.0__math.json


[LOAD] meta-llama/Llama-3.2-1B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[LOAD] done in 12.5s | VRAM 7.44 GB | attn=sdpa

=== TinyLlama/TinyLlama-1.1B-Chat-v1.0  |  MATH (max_new=1536) ===


MATH (bs=16, max_new=1536):   0%|          | 0/32 [00:00<?, ?it/s]

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


  accuracy:        0.20%  (n=500)
  truncation rate: 97.8%  (no \boxed found)
  mean_latency:    1247.4 ms/sample
  wall time:       624.9s
  peak_vram:       3.43 GB
  saved -> TinyLlama-1.1B-Chat-v1.0__math.json
[FREE] VRAM: 2.21 GB

[DONE] Part B total wall time: 69.9 min


---
## 5. Final summary — all 5 models × 3 benchmarks

Reads the updated JSONs from Drive and pivots into the final table. Saves to `summary_v5.csv` alongside the old `summary_final.csv` so you can diff them.


In [11]:
import pandas as pd

BENCHMARKS = ['gsm8k', 'math', 'bfcl']

rows = []
for model_id in ALL_MODELS:
    slug = model_id.split('/')[-1]
    for bench in BENCHMARKS:
        p = RESULTS_DIR / f'{slug}__{bench}.json'
        if p.exists():
            with open(p) as f: data = json.load(f)
            rows.append({
                'model': slug,
                'params_B': PARAMS_B.get(model_id),
                'benchmark': bench,
                'accuracy': data.get('accuracy'),
                'n': data.get('n'),
                'grader_version': data.get('grader_version', 'v4'),
                'max_new_tokens': data.get('max_new_tokens'),
                'truncation_rate': data.get('truncation_rate'),
            })
        else:
            rows.append({'model': slug, 'params_B': PARAMS_B.get(model_id),
                         'benchmark': bench, 'accuracy': None, 'n': None,
                         'grader_version': 'MISSING',
                         'max_new_tokens': None, 'truncation_rate': None})

df_long = pd.DataFrame(rows)
print('Long format:')
print(df_long.to_string(index=False))

df_pivot = df_long.pivot_table(
    index=['model', 'params_B'],
    columns='benchmark', values='accuracy').reset_index()
df_pivot = df_pivot.sort_values('params_B')
print('\nPivot (accuracy, 3 benchmarks, 5 models):')
print(df_pivot.to_string(index=False, float_format='{:.3f}'.format))

csv_path = RESULTS_DIR / 'summary_v5.csv'
df_pivot.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')


Long format:
                   model  params_B benchmark  accuracy    n      grader_version  max_new_tokens  truncation_rate
   Llama-3.2-1B-Instruct      1.23     gsm8k  0.396513 1319 v5_numeric_equality             NaN              NaN
   Llama-3.2-1B-Instruct      1.23      math  0.254000  500                  v5          1536.0            0.204
   Llama-3.2-1B-Instruct      1.23      bfcl  0.426667  600                  v4             NaN              NaN
   Llama-3.2-3B-Instruct      3.21     gsm8k  0.746020 1319 v5_numeric_equality             NaN              NaN
   Llama-3.2-3B-Instruct      3.21      math  0.414000  500                  v5          1536.0            0.150
   Llama-3.2-3B-Instruct      3.21      bfcl  0.893333  600                  v4             NaN              NaN
  Phi-3-mini-4k-instruct      3.82     gsm8k  0.880970 1319 v5_numeric_equality             NaN              NaN
  Phi-3-mini-4k-instruct      3.82      math  0.412000  500                  v5    

In [12]:
# Compare to Meta's published Llama 3.2 numbers
META_PUBLISHED = {
    'Llama-3.2-1B-Instruct': {'gsm8k': 0.444, 'math': 0.306, 'bfcl': 0.257},
    'Llama-3.2-3B-Instruct': {'gsm8k': 0.777, 'math': 0.480, 'bfcl': 0.670},
}

print('=== Reproducibility check vs Meta published numbers ===\n')
print(f'{"Model":30s} {"Bench":8s} {"Ours":>8s} {"Meta":>8s} {"Delta":>8s}')
print('-' * 70)
for model_slug, meta_nums in META_PUBLISHED.items():
    for bench, meta_acc in meta_nums.items():
        p = RESULTS_DIR / f'{model_slug}__{bench}.json'
        if not p.exists():
            continue
        with open(p) as f: data = json.load(f)
        ours = data.get('accuracy')
        if ours is None: continue
        delta = ours - meta_acc
        flag = ''
        if abs(delta) > 0.05:
            flag = '  <-- >5pp gap'
        print(f'{model_slug:30s} {bench:8s} {100*ours:>7.2f}% {100*meta_acc:>7.2f}% {100*delta:>+7.2f}pp{flag}')


=== Reproducibility check vs Meta published numbers ===

Model                          Bench        Ours     Meta    Delta
----------------------------------------------------------------------
Llama-3.2-1B-Instruct          gsm8k      39.65%   44.40%   -4.75pp
Llama-3.2-1B-Instruct          math       25.40%   30.60%   -5.20pp  <-- >5pp gap
Llama-3.2-1B-Instruct          bfcl       42.67%   25.70%  +16.97pp  <-- >5pp gap
Llama-3.2-3B-Instruct          gsm8k      74.60%   77.70%   -3.10pp
Llama-3.2-3B-Instruct          math       41.40%   48.00%   -6.60pp  <-- >5pp gap
Llama-3.2-3B-Instruct          bfcl       89.33%   67.00%  +22.33pp  <-- >5pp gap


In [13]:
# Disconnect runtime after the run completes
from google.colab import runtime
runtime.unassign()
